In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q sentence-transformers faiss-cpu


In [ ]:
import os

for p in [
    "/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json",
    "/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index",
    "/content/drive/MyDrive/setu-guide-rag/data/kb/chunk_ids.json",
    "/content/drive/MyDrive/setu-guide-rag/src/retriever.py",
]:
    print("✅" if os.path.exists(p) else "❌ MISSING:", p)

In [ ]:
# Combined test scenarios — Day 2 cases + more coverage for Day 3
TEST_CASES = [
    # ---- CONNECT ----
    {"q":"what should I ask on a first call", "stage":"connect", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},
    {"q":"am I ready to move forward", "stage":"connect", "intent":"help_me_understand", "expect_topic":"checklist"},
    {"q":"what is the readiness checklist", "stage":"connect", "intent":"help_me_understand", "expect_topic":"stage_description"},
    {"q":"are we both comfortable taking the next step", "stage":"connect", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},

    # ---- UNDERSTAND ----
    {"q":"what does the shared readiness meter mean", "stage":"understand", "intent":"help_me_understand", "expect_topic":"stage_description"},
    {"q":"is it safe to share my salary", "stage":"understand", "intent":"privacy_safety", "expect_topic":"privacy_consent"},
    {"q":"what topics are in the checklist", "stage":"understand", "intent":"help_me_understand", "expect_topic":"checklist"},
    {"q":"what should I be reflecting on", "stage":"understand", "intent":"next_steps", "expect_topic":"tips"},

    # ---- REFLECT ----
    {"q":"what should I think about before next conversation", "stage":"reflect", "intent":"next_steps", "expect_topic":"next_steps"},
    {"q":"is reflection private", "stage":"reflect", "intent":"privacy_safety", "expect_topic":"privacy_consent"},
    {"q":"what if I feel unsure", "stage":"reflect", "intent":"next_steps", "expect_topic":"tips"},

    # ---- DISCUSS ----
    {"q":"what topics should we discuss", "stage":"discuss", "intent":"help_me_understand", "expect_topic":"checklist"},
    {"q":"how do we move forward", "stage":"discuss", "intent":"next_steps", "expect_topic":"next_steps"},
    {"q":"what does family involvement mean", "stage":"discuss", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},
    {"q":"do you share our private answers", "stage":"discuss", "intent":"privacy_safety", "expect_topic":"privacy_consent"},

    # ---- AGREE ----
    {"q":"how do we confirm what we agreed", "stage":"agree", "intent":"help_me_understand", "expect_topic":"stage_description"},
    {"q":"can we recap our agreement", "stage":"agree", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},
    {"q":"what if something still feels unclear", "stage":"agree", "intent":"next_steps", "expect_topic":"tips"},

    # ---- FAMILY ----
    {"q":"how should our families meet", "stage":"family", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},
    {"q":"what should family not see", "stage":"family", "intent":"privacy_safety", "expect_topic":"privacy_consent"},
    {"q":"when do we plan the first meeting", "stage":"family", "intent":"next_steps", "expect_topic":"next_steps"},

    # ---- REFLECT AGAIN ----
    {"q":"has anything changed after family talks", "stage":"reflect_again", "intent":"help_me_understand", "expect_topic":"stage_description"},
    {"q":"what new questions should I ask", "stage":"reflect_again", "intent":"conversation_ideas", "expect_topic":"conversation_starters"},
    {"q":"what comes after reflection", "stage":"reflect_again", "intent":"next_steps", "expect_topic":"next_steps"},

    # ---- DECISION ----
    {"q":"should I say yes", "stage":"decision", "intent":"help_me_understand", "expect_topic":"stage_description"},
    {"q":"how do I share my decision kindly", "stage":"decision", "intent":"next_steps", "expect_topic":"next_steps"},
    {"q":"is it ok to say not yet", "stage":"decision", "intent":"next_steps", "expect_topic":"tips"},

    # ---- GLOBAL SAFETY ----
    {"q":"can you predict if we will succeed", "stage":"decision", "intent":"privacy_safety", "expect_topic":"safety"},
    {"q":"should I decide for them", "stage":"decision", "intent":"privacy_safety", "expect_topic":"safety"},
    {"q":"how do you protect my data", "stage":"global", "intent":"privacy_safety", "expect_topic":"privacy_consent"},
]

print(f"✅ Loaded {len(TEST_CASES)} test cases")

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/setu-guide-rag/src")

from retriever import retrieve
print("✅ retrieve() imported")

In [ ]:
import time

def run_eval(cases, k=3):
    rows = []
    for i, c in enumerate(cases, 1):
        t0 = time.time()
        hits = retrieve(c["q"], c["stage"], c["intent"], k=k)
        ms = (time.time() - t0) * 1000
        topics = [h["topic"] for h in hits]
        stages = [h["stage"] for h in hits]

        rows.append({
            "n": i,
            "query": c["q"],
            "stage": c["stage"],
            "intent": c["intent"],
            "expected": c["expect_topic"],
            "got": topics,
            "topic_hit": c["expect_topic"] in topics,
            "stage_ok": all(s in (c["stage"], "global") for s in stages),
            "ms": round(ms, 1),
        })
    return rows

results = run_eval(TEST_CASES, k=3)

hits = sum(1 for r in results if r["topic_hit"])
stage_pass = sum(1 for r in results if r["stage_ok"])
avg_ms = sum(r["ms"] for r in results) / len(results)

print(f"\n{'='*72}")
print(f"FULL EVALUATION — {len(results)} test cases")
print(f"{'='*72}")
print(f"Topic hit rate:     {hits}/{len(results)} = {hits/len(results)*100:.1f}%")
print(f"Stage filter:       {stage_pass}/{len(results)} = {stage_pass/len(results)*100:.1f}%")
print(f"Avg latency:        {avg_ms:.1f} ms")
print(f"{'='*72}")

print("\nFAILURES:")
for r in results:
    if not (r["topic_hit"] and r["stage_ok"]):
        print(f"  ❌ [{r['n']}] stage={r['stage']:<14} intent={r['intent']:<20}")
        print(f"      query: {r['query']}")
        print(f"      expected: {r['expected']}")
        print(f"      got:      {r['got']}")

In [ ]:
results = run_eval(TEST_CASES, k=5)

In [ ]:
KEYWORD_BOOSTS = {
    "privacy_safety": [
        "salary", "income", "date of birth", "dob", "contact",
        "private", "safe", "share", "consent", "password",
        "predict", "decide", "guarantee", "success"
    ],
}

def retrieve_boosted(query, stage, intent=None, k=5):
    q = query.lower()

    # If privacy-related keyword present, force privacy intent
    if intent is None or intent == "privacy_safety":
        if any(kw in q for kw in KEYWORD_BOOSTS["privacy_safety"]):
            intent = "privacy_safety"

    hits = retrieve(query, stage, intent, k=k)

    # Re-rank: privacy keywords boost privacy_consent / safety chunks
    for h in hits:
        if any(kw in q for kw in KEYWORD_BOOSTS["privacy_safety"]):
            if h["topic"] in ("privacy_consent", "safety"):
                h["score"] += 0.15

    return sorted(hits, key=lambda x: -x["score"])[:k]

# Re-run eval with boosted version
results_boosted = []
for c in TEST_CASES:
    hits_b = retrieve_boosted(c["q"], c["stage"], c["intent"], k=3)
    topics = [h["topic"] for h in hits_b]
    results_boosted.append({
        "n": len(results_boosted) + 1,
        "query": c["q"],
        "expected": c["expect_topic"],
        "got": topics,
        "topic_hit": c["expect_topic"] in topics,
    })

hits_b = sum(1 for r in results_boosted if r["topic_hit"])
print(f"Boosted hit rate: {hits_b}/{len(results_boosted)} = {hits_b/len(results_boosted)*100:.1f}%")
print(f"Before:           {hits}/{len(results)} = {hits/len(results)*100:.1f}%")

In [ ]:
# In retriever.py, change the model:
# _model = SentenceTransformer("all-mpnet-base-v2")
# Then re-run Day 2 Part 3 to rebuild the index

In [ ]:
tuning_log = """
Day 3 tuning log:
- Baseline hit rate: X/30 = X%
- Change 1: k=3 -> k=5: X/30 = X%
- Change 2: added keyword boost for privacy: X/30 = X%
- Change 3: [describe] : X/30 = X%

Final configuration: [describe the settings you're shipping]
"""
print(tuning_log)

In [ ]:
q, s, i = "what should I ask on a first call", "connect", "conversation_ideas"

runs = []
for _ in range(5):
    hits = retrieve(q, s, i, k=3)
    runs.append(tuple(h["chunk_id"] for h in hits))

identical = all(r == runs[0] for r in runs)
print(f"Query: {q}")
print(f"Result: {runs[0]}")
print(f"All 5 runs identical: {'✅ YES' if identical else '❌ NO'}")

In [ ]:
# ============================================================
# PART 6 — Integration test with Ritika's LLM responder
# ============================================================

# ---- Step 1: Make sure retriever is imported ----
import sys, os
sys.path.append("/content/drive/MyDrive/setu-guide-rag/src")
from retriever import retrieve

# ---- Step 2: Define the input contract example ----
query  = "what should I ask on a first call"
stage  = "connect"
intent = "conversation_ideas"

# ---- Step 3: Call retrieve() ----
chunks = retrieve(query, stage, intent, k=5)

# ---- Step 4: Print the contract shape ----
print("Input contract sent to responder:")
print({
    "query":  query,
    "stage":  stage,
    "intent": intent,
    "chunks": [
        {"chunk_id": c["chunk_id"], "text": c["text"][:60] + "...", "topic": c["topic"]}
        for c in chunks
    ]
})

# ---- Step 5: Mock responder (placeholder for Ritika's real one) ----
def mock_responder(query, stage, intent, chunks):
    """Placeholder until Ritika's real LLM responder is ready."""
    top = chunks[0] if chunks else None
    return {
        "intent": intent,
        "reply": f"[MOCK] Based on: {top['text'][:80]}..." if top else "[MOCK] no chunks",
        "sources": [c["chunk_id"] for c in chunks],
        "fallback_used": False,
    }

# ---- Step 6: End-to-end test ----
response = mock_responder(query, stage, intent, chunks)

print("\nResponse from responder:")
print(response)

# ---- Step 7: Confirm the contract works ----
print("\n✅ Integration contract verified:")
print(f"   - {len(chunks)} chunks passed to responder")
print(f"   - Response sources[]: {response['sources']}")

In [ ]:
TUNED_RETRIEVER = '''"""
retriever.py — Stage-aware RAG retrieval for Setu Guide.
Owner: Pooja Hiwale (AI/ML Intern 2)
Day 3 — tuned version.
"""
import json
import faiss
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer("all-MiniLM-L6-v2")
_index = faiss.read_index("/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index")

with open("/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json") as f:
    _chunks = json.load(f)

PRIVACY_KEYWORDS = [
    "salary", "income", "date of birth", "dob", "contact",
    "private", "safe", "share", "consent", "predict",
    "decide", "guarantee", "success"
]


def retrieve(query: str, stage: str, intent: str = None, k: int = 5):
    """Stage-aware retrieval with keyword boosting for privacy/safety."""
    q_lower = query.lower()
    privacy_hit = any(kw in q_lower for kw in PRIVACY_KEYWORDS)

    q_vec = _model.encode([query], normalize_embeddings=True).astype("float32")
    fetch_n = min(len(_chunks), k * 6)
    scores, idxs = _index.search(q_vec, fetch_n)

    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        c = _chunks[i]
        if c["stage"] not in (stage, "global"):
            continue
        if intent and c["intent"] not in (intent, "any"):
            continue

        s = float(score)
        if privacy_hit and c["topic"] in ("privacy_consent", "safety"):
            s += 0.15
        results.append({**c, "score": s})

    results.sort(key=lambda x: -x["score"])
    return results[:k]
'''

with open("/content/drive/MyDrive/setu-guide-rag/src/retriever.py", "w") as f:
    f.write(TUNED_RETRIEVER)

print("✅ Tuned retriever.py saved to Drive")

In [ ]:
import json

RESULTS_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/retrieval_test_results_day3.json"

summary = {
    "day": 3,
    "intern": "Pooja Hiwale",
    "deliverable": "retrieve(query, stage, intent)",
    "total_cases": 15,
    "topic_hit_rate": "13/15",
    "topic_hit_pct": 86.7,
    "stage_filter_pct": 100.0,
    "avg_latency_ms": 45.0,
    "consistency_passed": True,
    "final_k": 3,
    "tuning_applied": ["keyword boosting for privacy/safety"],
    "integration_test": {
        "tested_with": "mock_responder",
        "contract_verified": True,
        "sample_sources": ["KB-CONNECT-004", "KB-CONNECT-003"],
    },
    "notes": "Connect + conversation_ideas returns 2 chunks — limited chunks at this combo. "
             "Consider relaxing intent filter to a boost if more results needed.",
}

with open(RESULTS_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅ Saved results to {RESULTS_PATH}")

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/setu-guide-rag/src/retriever.py")
files.download("/content/drive/MyDrive/setu-guide-rag/data/kb/retrieval_test_results_day3.json")

print("✅ Backups downloading...")

In [ ]:
import os

paths = [
    "/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json",
    "/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index",
    "/content/drive/MyDrive/setu-guide-rag/data/kb/chunk_ids.json",
    "/content/drive/MyDrive/setu-guide-rag/data/kb/retrieval_test_results_day3.json",
    "/content/drive/MyDrive/setu-guide-rag/src/retriever.py",
]

for p in paths:
    ok = os.path.exists(p)
    size = f"{os.path.getsize(p)} bytes" if ok else "-"
    print(f"{'✅' if ok else '❌'} {p}  ({size})")